# Análise de Resultados - Modelo REPLAN Validation Paper 01
Este notebook foi gerado automaticamente para carregar o modelo de engrenamento salvo via `pickle`, permitindo a recuperação de dados, plotagem de dashboards e extração de séries temporais para análise detalhada de dinâmica de rotores com folga (*backlash*).

In [ ]:
# 1. Importações e Configurações de Ambiente
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import sys

# IMPORTANTE: Certifique-se de que o arquivo 'backlash.py' está na mesma pasta que este notebook.
try:
    from backlash import Backlash
    print("Classe Backlash importada com sucesso!")
except ImportError:
    print("ERRO: Não foi possível encontrar o arquivo 'backlash.py'. Coloque-o no mesmo diretório deste notebook.")

## 2. Carregamento do Modelo Binário (`.pkl`)
Vamos ressuscitar o objeto utilizando o método estático `load_model` nativo da sua classe, sem a necessidade de remontar a estrutura de rotores e discos do ROSS.

In [ ]:
# Define o caminho para o arquivo .pkl
caminho_arquivo = "modelo_completo_REPLAN_validation_paper_01.pkl"

if os.path.exists(caminho_arquivo):
    print(f"Carregando o modelo de '{caminho_arquivo}'...")
    modelo = Backlash.load_model(caminho_arquivo)
    
    # Exibe informações básicas do modelo carregado
    print("\n=== INFORMAÇÕES DO MODELO CARREGADO ===")
    print(f"Velocidade do Pinhão : {modelo.speed_driving_gear:.2f} rad/s ({modelo.speed_driving_gear * 30 / np.pi:.0f} RPM)")
    print(f"Número de Pontos      : {len(modelo.time)}")
    print(f"Backlash Inicial (b0) : {modelo.b0:.4e} m")
    print(f"Amplitude de Erro     : {modelo.error_amp:.4e} m")
else:
    print(f"ERRO: O arquivo '{caminho_arquivo}' não foi encontrado na pasta atual.")
    print("Por favor, mova o arquivo .pkl para este mesmo diretório ou ajuste a variável 'caminho_arquivo'.")

## 3. Execução dos Dashboards Nativos da Classe
A classe `Backlash` possui métodos embutidos de alta performance acelerados por Numba para processar e exportar os dashboards completos em HTML interativo.

In [ ]:
# 3.1. Gerar o Dashboard Principal (Tempo vs Frequência / FFT)
# O parâmetro 'decimation' ajuda a reduzir o peso do gráfico se houver muitos pontos (ex: decimation=10 plota 1 a cada 10 pontos)
print("Gerando Dashboard Principal em HTML...")
modelo.plot_dashboard(
    freq_unit="Hz", 
    decimation=1, 
    dft_y_scale="log",
    save_path="dashboard_replan_completo.html"
)

In [ ]:
# 3.2. Gerar o Mapa de Poincaré e Espaço de Fase
# Este comando vai gerar dois arquivos HTML separados (um para o pinhão/DTE e outro para a coroa)
# e exportará um arquivo CSV com as amostras exatas sincronizadas pelo período de engrenamento.
print("Gerando Mapas de Poincaré e Espaço de Fase...")
modelo.plot_poincare_map(
    is_linear=False, 
    plot_filename="mapa_poincare_replan.html", 
    csv_filename="dados_poincare_replan.csv",
    use_spline=True  # Ativa interpolação por Spline Cúbica para derivadas analíticas perfeitas
)

## 4. Extração de Dados e Análises Customizadas com Pandas
Caso queira realizar pós-processamentos personalizados que não estão nos plots nativos (como cálculo de RMS, curtose, ou filtros adicionais), podemos extrair os arrays brutos do objeto e convertê-los em um DataFrame do Pandas.

In [ ]:
# Extraindo as séries temporais do dicionário de resultados do backlash
tempo = modelo.time
resultados = modelo.backlash_results

# Construindo o DataFrame com as principais grandezas físicas
df = pd.DataFrame({
    "Tempo_s": tempo,
    "DTE_um": resultados["delta"] * 1e6,          # Erro de Transmissão Dinâmico em micrômetros
    "Folga_bt_um": resultados["bt"] * 1e6,        # Limite da folga dinâmica em micrômetros
    "Forca_Mesh_N": resultados["Fm"],             # Força normal de engrenamento em Newtons
    "Rigidez_Nm": resultados["K_time"],           # Rigidez de engrenamento instantânea
    "Razao_Contato_CR": resultados["contact_ratio"] # Razão de contato dinâmica
})

# Exibe as primeiras linhas do DataFrame
print("Visualização das primeiras linhas dos dados estruturados:")
display(df.head())

# Exporta para um arquivo CSV unificado caso queira abrir no Excel
df.to_csv("dados_extraidos_replan.csv", index=False)
print("\nDados temporais brutos exportados com sucesso para 'dados_extraidos_replan.csv'!")

In [ ]:
# 4.2. Gráfico Customizado Sobreposto: DTE vs Limites da Folga
fig_custom = go.Figure()

# Adiciona o DTE dinâmico
fig_custom.add_trace(go.Scattergl(x=df["Tempo_s"], y=df["DTE_um"], name="DTE (δ)", line=dict(color='blue', width=1.5)))

# Adiciona os limites superior e inferior da folga
fig_custom.add_trace(go.Scattergl(x=df["Tempo_s"], y=df["Folga_bt_um"], name="+bt (Impacto Frente)", line=dict(color='red', width=1, dash='dash')))
fig_custom.add_trace(go.Scattergl(x=df["Tempo_s"], y=-df["Folga_bt_um"], name="-bt (Impacto Dorso)", line=dict(color='red', width=1, dash='dash')))

fig_custom.update_layout(
    title=f"Comportamento do DTE dentro da Folga Dinâmica ({modelo.speed_driving_gear * 30 / np.pi:.0f} RPM)",
    xaxis_title="Tempo (s)",
    yaxis_title="Deslocamento (µm)",
    template="plotly_white",
    hovermode="x unified"
)

fig_custom.show()

In [ ]:
# 4.3. Estatísticas Métricas de Vibração de Engrenagens
print("=== MÉTRICAS ESTATÍSTICAS DA SIMULAÇÃO ===")
print(f"Força Máxima de Impacto na Malha (Fm) : {df['Forca_Mesh_N'].max():.2f} N")
print(f"Força Mínima de Engrenamento (Fm)   : {df['Forca_Mesh_N'].min():.2f} N")
print(f"Valor Médio da Força de Engrenamento : {df['Forca_Mesh_N'].mean():.2f} N")
print(f"DTE Médio                            : {df['DTE_um'].mean():.4f} µm")
print(f"DTE Máximo                           : {df['DTE_um'].max():.4f} µm")
print(f"DTE RMS (Root Mean Square)           : {np.sqrt(np.mean(df['DTE_um']**2)):.4f} µm")

# Identificação de descolamento de dentes (se Fm chegar a zero)
pontos_zero_forca = (df['Forca_Mesh_N'] <= 1e-3).sum()
percentual_descolamento = (pontos_zero_forca / len(df)) * 100
print(f"Percentual de tempo em descolamento  : {percentual_descolamento:.2f}%")